<a href="https://colab.research.google.com/github/RoihansLab/Data-Mining-Project/blob/main/youtube_comment_sentiment/Classifcation/Classifcation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
!pip install scikit-optimize

import pandas as pd
import numpy as np
import re
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score
from imblearn.over_sampling import SMOTE
from skopt import BayesSearchCV
from skopt.space import Real, Categorical, Integer
import time
from skopt import BayesSearchCV
from skopt.space import Real, Categorical, Integer
import time

# Kunci Random State agar hasil selalu sama setiap di-run
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("✅ Cell 1 Selesai: Semua library berhasil dimuat dan Random State dikunci!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.8/107.8 kB 4.1 MB/s eta 0:00:00
✅ Cell 1 Selesai: Semua library berhasil dimuat dan Random State dikunci!


In [18]:
# Baca dataset hasil Qwen
df = pd.read_csv("/content/hasil_dari_LLM_qwen.csv")
df = df.dropna(subset=['comment'])

# Encode kolom prediksi_ai menjadi angka 0, 1, 2
le = LabelEncoder()
df['target'] = le.fit_transform(df['prediksi_ai'])

mapping_label = dict(zip(le.classes_, le.transform(le.classes_)))
print(f"Mapping Label: {mapping_label}")

# Tampilkan 5 data teratas untuk memastikan
display(df[['video_title', 'comment', 'prediksi_ai', 'target']].head())

Mapping Label: {'negatif': np.int64(0), 'netral': np.int64(1), 'positif': np.int64(2)}


,video_title,comment,prediksi_ai,target
0,Pekerja Latih Robot Untuk Gantikan Mereka! KIA...,Bahas imigrant maroko yg nyebrang ke spanyol d...,netral,1
1,Pekerja Latih Robot Untuk Gantikan Mereka! KIA...,Menurut saya jika semua pengusaha berpikiran s...,negatif,0
2,Pekerja Latih Robot Untuk Gantikan Mereka! KIA...,jujur ya sebagai seorang yang bekerja di bidan...,negatif,0
3,Pekerja Latih Robot Untuk Gantikan Mereka! KIA...,Gw sebagai illustrator susah bener dapet klien...,negatif,0
4,Pekerja Latih Robot Untuk Gantikan Mereka! KIA...,Niat untung -> efesiensi phk->produksi meningk...,negatif,0


In [19]:
# Gabungkan judul dan komen
df['teks_gabungan'] = df['video_title'].fillna('') + " " + df['comment'].fillna('')

# Fungsi untuk huruf kecil dan buang simbol
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', '', text)
    return text

df['clean_text'] = df['teks_gabungan'].apply(clean_text)

print("✅ Teks berhasil dibersihkan! Ini perbandingan sebelum vs sesudah:")
display(df[['teks_gabungan', 'clean_text']].head())

✅ Teks berhasil dibersihkan! Ini perbandingan sebelum vs sesudah:


,teks_gabungan,clean_text
0,Pekerja Latih Robot Untuk Gantikan Mereka! KIA...,pekerja latih robot untuk gantikan mereka kiam...
1,Pekerja Latih Robot Untuk Gantikan Mereka! KIA...,pekerja latih robot untuk gantikan mereka kiam...
2,Pekerja Latih Robot Untuk Gantikan Mereka! KIA...,pekerja latih robot untuk gantikan mereka kiam...
3,Pekerja Latih Robot Untuk Gantikan Mereka! KIA...,pekerja latih robot untuk gantikan mereka kiam...
4,Pekerja Latih Robot Untuk Gantikan Mereka! KIA...,pekerja latih robot untuk gantikan mereka kiam...


In [20]:
X_raw = df['clean_text']
y = df['target']

# Split 80% Train, 20% Test
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

# TF-IDF Vectorizer
tfidf = TfidfVectorizer(max_features=3000, ngram_range=(1, 2))
X_train_tfidf = tfidf.fit_transform(X_train_raw)
X_test_tfidf = tfidf.transform(X_test_raw) # Ingat: test hanya di-transform

print(f"Jumlah Data Latih (Train): {X_train_tfidf.shape[0]} baris")
print(f"Jumlah Data Uji (Test): {X_test_tfidf.shape[0]} baris")
print(f"Jumlah Fitur TF-IDF (Kosakata unik): {X_train_tfidf.shape[1]} kolom")

Jumlah Data Latih (Train): 3246 baris
Jumlah Data Uji (Test): 812 baris
Jumlah Fitur TF-IDF (Kosakata unik): 1500 kolom


In [21]:
print("📉 Keseimbangan Kelas Data Latih SEBELUM SMOTE:")
print(y_train.value_counts().sort_index())

# Terapkan SMOTE
smote = SMOTE(random_state=RANDOM_STATE)
X_train_smote, y_train_smote = smote.fit_resample(X_train_tfidf, y_train)

print("\n📈 Keseimbangan Kelas Data Latih SETELAH SMOTE:")
print(y_train_smote.value_counts().sort_index())

📉 Keseimbangan Kelas Data Latih SEBELUM SMOTE:
target
0     627
1    2523
2      96
Name: count, dtype: int64

📈 Keseimbangan Kelas Data Latih SETELAH SMOTE:
target
0    2523
1    2523
2    2523
Name: count, dtype: int64


In [28]:
models = {
    "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE, class_weight='balanced'),
    "Naive Bayes": MultinomialNB(), # Naive Bayes tidak punya parameter class_weight
    "Random Forest": RandomForestClassifier(random_state=RANDOM_STATE, class_weight='balanced'),
    "Logistic Regression": LogisticRegression(random_state=RANDOM_STATE, max_iter=1000, class_weight='balanced')
}

base_results = {}
print("=== HASIL EVALUASI MODEL LENGKAP (SEBELUM TUNING) ===")

for name, model in models.items():
    # Latih model
    model.fit(X_train_smote, y_train_smote)

    # Prediksi
    y_pred = model.predict(X_test_tfidf)

    print(f"\n======================================================")
    print(f"📊 REPORT: {name.upper()}")
    print("======================================================")
    # Tampilkan Precision, Recall, dan F1-Score per kelas
    print(classification_report(y_test, y_pred, target_names=['Negatif', 'Netral', 'Positif'], digits=4))

    # Simpan F1-Macro untuk perbandingan di grafik nanti
    f1_mac = f1_score(y_test, y_pred, average='macro')
    base_results[name] = {'F1-Macro': f1_mac}

=== HASIL EVALUASI MODEL LENGKAP (SEBELUM TUNING) ===

📊 REPORT: DECISION TREE
              precision    recall  f1-score   support

     Negatif     0.3929    0.4204    0.4062       157
      Netral     0.8488    0.8098    0.8289       631
     Positif     0.0952    0.1667    0.1212        24

    accuracy                         0.7155       812
   macro avg     0.4456    0.4656    0.4521       812
weighted avg     0.7384    0.7155    0.7262       812


📊 REPORT: NAIVE BAYES
              precision    recall  f1-score   support

     Negatif     0.3860    0.6688    0.4895       157
      Netral     0.9045    0.6006    0.7219       631
     Positif     0.0413    0.2083    0.0690        24

    accuracy                         0.6022       812
   macro avg     0.4440    0.4926    0.4268       812
weighted avg     0.7788    0.6022    0.6577       812


📊 REPORT: RANDOM FOREST
              precision    recall  f1-score   support

     Negatif     0.5217    0.4586    0.4881       157
  

In [29]:
search_spaces = {
    "Decision Tree": {'max_depth': Integer(5, 50), 'min_samples_split': Integer(2, 20)},
    "Naive Bayes": {'alpha': Real(0.01, 10.0, prior='log-uniform')},
    "Random Forest": {'n_estimators': Integer(50, 200), 'max_depth': Integer(5, 50)},
    "Logistic Regression": {'C': Real(0.01, 100.0, prior='log-uniform'), 'solver': Categorical(['lbfgs', 'liblinear'])}
}

tuned_results = {}
best_models = {}

print("⏳ Sedang memproses Bayesian Hyperparameter Tuning...")

for name, model in models.items():
    start_time = time.time()

    # Tuning Model
    bayes_search = BayesSearchCV(
        estimator=model, search_spaces=search_spaces[name],
        n_iter=15, cv=3, scoring='f1_macro', n_jobs=-1, random_state=RANDOM_STATE
    )
    bayes_search.fit(X_train_smote, y_train_smote)

    # Prediksi menggunakan model terbaik
    best_model = bayes_search.best_estimator_
    best_models[name] = best_model
    y_pred = best_model.predict(X_test_tfidf)

    end_time = time.time()

    print(f"\n======================================================")
    print(f"🎯 REPORT SETELAH TUNING: {name.upper()}")
    print(f"⏱️ Waktu Tuning: {(end_time - start_time):.2f} detik")
    print(f"🔧 Parameter Terbaik: {bayes_search.best_params_}")
    print("======================================================")
    # Tampilkan Precision, Recall, dan F1-Score per kelas
    print(classification_report(y_test, y_pred, target_names=['Negatif', 'Netral', 'Positif'], digits=4))

    # Simpan hasil untuk grafik
    f1_mac = f1_score(y_test, y_pred, average='macro')
    tuned_results[name] = {'F1-Macro': f1_mac, 'Best_Params': bayes_search.best_params_}

⏳ Sedang memproses Bayesian Hyperparameter Tuning...

🎯 REPORT SETELAH TUNING: DECISION TREE
⏱️ Waktu Tuning: 51.03 detik
🔧 Parameter Terbaik: OrderedDict({'max_depth': 46, 'min_samples_split': 2})
              precision    recall  f1-score   support

     Negatif     0.4211    0.4076    0.4142       157
      Netral     0.8454    0.8320    0.8387       631
     Positif     0.1026    0.1667    0.1270        24

    accuracy                         0.7303       812
   macro avg     0.4563    0.4688    0.4600       812
weighted avg     0.7414    0.7303    0.7356       812


🎯 REPORT SETELAH TUNING: NAIVE BAYES
⏱️ Waktu Tuning: 4.48 detik
🔧 Parameter Terbaik: OrderedDict({'alpha': 0.010253943538922507})
              precision    recall  f1-score   support

     Negatif     0.3707    0.6115    0.4615       157
      Netral     0.8686    0.6498    0.7434       631
     Positif     0.0247    0.0833    0.0381        24

    accuracy                         0.6256       812
   macro avg     